# 12 · Select a device across simulation, learning and chemistry

These APIs require the current source checkout. Core CPU work needs NumPy;
Torch CPU/CUDA is optional. Install a compatible CUDA-enabled Torch build using
the official PyTorch selector, then install this checkout with `.[gpu,tutorial]`.
`python -m nexqus.devices` reports the actual hardware/build. The earlier image
lesson's CPU-index install cannot execute CUDA. See `docs/GPU.md`.

This lesson uses CUDA when available and explicitly records a CUDA skip otherwise.
Torch CPU comparisons still run without a GPU. A CPU pass is not CUDA evidence.
No cloud resource or QPU is used.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.4.0rc1', 'Install NexQus SDK 1.4.0rc1 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


In [ ]:
import importlib.util
torch_available = importlib.util.find_spec('torch') is not None
if torch_available:
    import torch
    torch.set_num_threads(1)
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    if device == 'cpu': print('SKIPPED optional CUDA: no CUDA device; testing Torch CPU')
    sim = Simulator.for_device(device, backend='torch')
    print('Actual execution:', sim.backend.name, sim.backend.device)
else:
    print('SKIPPED optional CUDA: install a compatible Torch build to run this lesson')


## Native results, noise and dynamic circuits

`native_result=True` keeps retained states and result tensors on the simulation
device. Pauli expectation reduction runs there too; the returned Python float
is a scalar export. `to_numpy()` explicitly exports a host result for plotting.
Noise and dynamic circuits keep quantum evolution on the selected device,
while classical records/branch bookkeeping remain Python host work.


In [ ]:
if torch_available:
    q = Circuit(2).h(0).cx(0, 1)
    result = sim.run(q, shots=0, native_result=True)
    assert result.probabilities.device.type == torch.device(device).type
    assert abs(result.expectation('ZZ')-1) < 1e-12
    np.testing.assert_allclose(result.to_numpy().probabilities, [.5,0,0,.5], atol=1e-12)
    noisy = sim.run(q, shots=0, noise=NoiseModel(Noise.amplitude_damping(.1)), native_result=True)
    assert abs(float(noisy.probabilities.sum())-1) < 1e-12
    dynamic = sim.run(Circuit(2).h(0).measure(0,'m').x(1).c_if('m',1), shots=0, native_result=True)
    np.testing.assert_allclose(dynamic.to_numpy().probabilities, [.5,0,0,.5], atol=1e-12)
    print('Native results, noise and feed-forward passed')


## Batched circuit autograd

The optional `TorchCircuit` module accepts shared scalar weights and per-sample
angle vectors. Its state and derivatives stay on the module's device. Static
RX/RY/RZ/P/CP expressions, constant unitaries, diagonals and permutations are
supported. Noise/dynamic execution uses Simulator instead.


In [ ]:
if torch_available:
    from nexqus.torch_circuit import TorchCircuit
    x = Parameter('x')
    layer = TorchCircuit(Circuit(2).ry(x,0).cx(0,1), device=device)
    angles = torch.tensor([.2,.5,.8], dtype=torch.float64, device=device, requires_grad=True)
    probabilities = layer({'x': angles})
    probabilities[:,0].sum().backward()
    torch.testing.assert_close(angles.grad, -.5*torch.sin(angles))
    print('Gradient device:', angles.grad.device)
    plt.plot(angles.detach().cpu(), probabilities[:,0].detach().cpu(), marker='o')
    plt.xlabel('Angle'); plt.ylabel('P(00)'); plt.show()


## H2 VQE and exact references

The fixed four-qubit Hamiltonian needs no PySCF, including on native Windows.
Both VQE evolution/Pauli reduction and the bounded exact reference can use CUDA.
The small parameter-dictionary optimizer is host control. New molecular RHF
can separately use GPU4PySCF on Linux/WSL; AO/MO integrals and Jordan-Wigner
mapping remain CPU. A `device` option for VQE does not relabel those CPU stages.


In [ ]:
if torch_available:
    from nexqus import Adam
    from nexqus.chemistry import h2_reference_problem, h2_pair_ansatz
    problem = h2_reference_problem()
    energy = problem.hamiltonian.function(h2_pair_ansatz(Parameter('theta')), simulator=sim)
    parameters = {'theta': 0.}; optimizer = Adam(.04)
    for _ in range(180):
        parameters = optimizer.step(parameters, energy.gradient(parameters))
    reference = problem.exact_energy(simulator=sim)
    assert abs(energy(parameters)-reference) < 2e-7
    print('VQE:', energy(parameters), 'Exact reference:', reference, 'Device:', sim.backend.device)


## Use the same selection elsewhere

`QuantumFunction(..., device='cuda:0')`, `QuantumDistribution(...,
device='cuda:0')`, `hamiltonian.function(..., device='cuda:0')` and
`ImageGAN(generator, device='cuda:0')` select CUDA explicitly. Shor accepts
`simulator=sim`; templates/QFT/Grover build ordinary circuits run by that simulator.
The main examples accept `--device cuda:0`. For default simulator calls set
`NEXQUS_DEVICE` in your terminal before starting Python/Jupyter. Explicit CPU
choices override it. Only `device='auto'` permits selecting CPU when CUDA is absent.

For real hardware validation run `python scripts/check_gpu_workflows.py` on
your GPU machine. It fails if CUDA is absent or tests skip/fail, and records
the actual GPU and test evidence. Add `--include-scf` to require live GPU4PySCF
RHF parity. GPU acceleration does not establish improved image-generation quality.

**Try it:** compare CPU/CUDA energies and gradients with the same precision,
then inspect exported device metadata rather than inferring execution from time.
